# 第7章（续）：自定义训练循环

**学习路线：**

| 阶段 | 章节 | 内容 | 关键词 |
|------|------|------|--------|
| **一、为什么自定义** | §8 | 内置 fit() 的局限性 | 自定义动机 |
| **二、手动训练循环** | §9 | GradientTape、梯度计算、权重更新 | tf.GradientTape |
| **三、性能优化** | §10 | tf.function 编译加速 | @tf.function |
| **四、轻量自定义** | §11 | 覆盖 train_step() | Model.train_step |

> **前置要求**：请先学完 `7.1-7.3 模型构建与内置训练.ipynb`。
>
> 本笔记本使用 MNIST 真实数据集，所有代码可直接运行。

In [ ]:
# === 基础导入 ===
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from matplotlib import pyplot as plt

print(f"TensorFlow: {tf.__version__}")

---
# 7.4 编写自定义的训练循环和评估循环

## §8 何时需要自定义训练循环

在 90% 的场景下，内置的 `fit()` 已经够用。
但当你需要**完全控制训练逻辑**时，就需要自定义训练循环。

| 场景 | 内置 fit() 能搞定？ | 需要？ |
|------|---------------------|--------|
| 标准分类/回归 | ✅ 能 | compile + fit |
| 多任务加权损失 | ✅ 能 | 自定义 loss 或 train_step |
| 对抗训练（GAN） | ❌ 不能 | 完全自定义循环 |
| 梯度裁剪/梯度惩罚 | ⚠️ 部分能 | 自定义 train_step |
| 元学习/双层优化 | ❌ 不能 | 完全自定义循环 |

### 典型自定义训练循环的核心步骤

```
1. 在梯度带（GradientTape）中运行前向传播，得到当前批量的损失值
2. 从 tape 中检索损失相对于模型权重的梯度
3. 用优化器更新模型权重以降低当前批量的损失值
```

### 三种自定义训练的层级关系

```text
  最灵活 ┌──────────────────────┐
         │  完全手写训练循环     │ ← GradientTape + 手动 apply_gradients
         │  (本节 §9-§10)        │
         ├──────────────────────┤
         │  覆盖 train_step()    │ ← 保留 fit() 的 callbacks 等功能
         │  (本节 §11)           │
         ├──────────────────────┤
  最简单  │  内置 fit()           │ ← compile + fit 一条龙
         └──────────────────────┘
```

---
# 第一阶段：完全自定义训练循环

从零开始手写训练循环，完全掌控每一个步骤。

## §9 完全自定义训练循环（MNIST 示例）

### 9.1 ~ 9.3 准备数据和模型

In [ ]:
# 9.1 加载 MNIST 数据
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# reshape 并归一化
x_train = x_train.reshape((60000, 28 * 28)).astype("float32") / 255
x_test = x_test.reshape((10000, 28 * 28)).astype("float32") / 255

# 划分训练集和验证集
x_val = x_train[:10000]
y_val = y_train[:10000]
x_train_small = x_train[10000:]
y_train_small = y_train[10000:]

print(f"训练集: {x_train_small.shape[0]} 样本, 形状 {x_train_small.shape}")
print(f"验证集: {x_val.shape[0]} 样本, 形状 {x_val.shape}")
print(f"测试集:  {x_test.shape[0]} 样本, 形状 {x_test.shape}")

In [ ]:
# 9.2 构建模型和优化器
# 注意：这里不用 compile()，因为我们不使用 fit()
model = keras.Sequential([
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax")
])

# 手动创建优化器和损失函数
optimizer = keras.optimizers.RMSprop(learning_rate=1e-3)
loss_fn = keras.losses.SparseCategoricalCrossentropy()

# 手动创建监控指标
train_acc_metric = keras.metrics.SparseCategoricalAccuracy()
val_acc_metric = keras.metrics.SparseCategoricalAccuracy()

In [ ]:
# 9.3 准备数据管道（tf.data）
batch_size = 64

train_ds = tf.data.Dataset.from_tensor_slices((x_train_small, y_train_small))
train_ds = train_ds.shuffle(buffer_size=1024).batch(batch_size)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(batch_size)

# 查看数据管道的结构
for xb, yb in train_ds.take(1):
    print(f"一个 batch: x={xb.shape}, y={yb.shape}")

### 9.4 自定义训练循环

核心流程：
```
for epoch in range(epochs):
    for batch_x, batch_y in train_ds:
        with tf.GradientTape() as tape:       # 1. 记录梯度
            logits = model(batch_x)            # 2. 前向传播
            loss = loss_fn(batch_y, logits)    # 3. 计算损失
        grads = tape.gradient(loss, weights)   # 4. 反向传播求梯度
        optimizer.apply_gradients(...)          # 5. 更新权重
        train_metric.update_state(y, logits)    # 6. 更新指标
    
    for val_x, val_y in val_ds:                # 7. 验证阶段
        val_logits = model(val_x, training=False)
        val_metric.update_state(val_y, val_logits)
    
    打印指标 → 重置指标
```

In [ ]:
# 9.4 完全自定义训练循环
epochs = 3

for epoch in range(epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{epochs}")
    print(f"{'='*50}")
    
    # ---- 训练阶段 ----
    for step, (batch_x, batch_y) in enumerate(train_ds):
        # 1. GradientTape 记录前向传播过程
        #    tape 会自动追踪所有 TensorFlow 操作，用于之后反向传播
        with tf.GradientTape() as tape:
            # 前向传播：通过模型得到预测 logits
            # training=True 表示开启 Dropout（训练模式）
            logits = model(batch_x, training=True)
            # 计算当前 batch 的损失值
            loss_value = loss_fn(batch_y, logits)
        
        # 2. 从 tape 中获取梯度
        #    计算损失对每个可训练权重的偏导数
        grads = tape.gradient(loss_value, model.trainable_weights)
        
        # 3. 用优化器更新权重
        #    将梯度应用到权重上，沿着梯度反方向更新
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        
        # 4. 更新训练精度指标（累积统计）
        train_acc_metric.update_state(batch_y, logits)
        
        # 每 100 步打印一次当前 loss
        if step % 100 == 0:
            print(f"  Step {step:4d}: loss = {loss_value:.4f}")
    
    # 打印本 epoch 结束时的训练精度
    train_acc = train_acc_metric.result()
    print(f"\n  训练精度: {train_acc:.4f}")
    train_acc_metric.reset_state()  # 重置指标，为下一个 epoch 准备
    
    # ---- 验证阶段 ----
    # 注意：training=False 表示关闭 Dropout（推理模式）
    for val_x, val_y in val_ds:
        val_logits = model(val_x, training=False)
        val_acc_metric.update_state(val_y, val_logits)
    
    val_acc = val_acc_metric.result()
    print(f"  验证精度: {val_acc:.4f}")
    val_acc_metric.reset_state()  # 重置指标

### 关键概念：`tf.GradientTape` 是什么？

`GradientTape` 是 TensorFlow 的**自动微分引擎**。

```text
普通代码：    y = f(x)     → 只计算结果
with tape:   y = f(x)     → 记录计算过程
             grads = tape.gradient(y, x)  → 根据记录反向传播求梯度
```

类比：就像在纸上写下一道数学题的每一步推导过程，
然后通过链式法则从后往前倒推每一步的偏导数。

---
## §10 利用 tf.function 加快运行速度

上面的纯 Python 训练循环虽然工作正常，但速度较慢。
因为每一步都要经过 Python 解释器。

**解决方案**：用 `@tf.function` 将训练步骤编译成计算图。

```text
普通 Python 代码：逐行解释执行（慢）
     ↓ @tf.function
编译成计算图：TensorFlow 追踪操作，构建静态图
     ↓
全局优化：算子融合、内存优化、并行计算
     ↓
直接执行编译好的图（快）
```

**注意**：第一次调用时需要编译（有开销），后续调用直接使用编译好的图。

In [ ]:
# 10.1 用 @tf.function 装饰训练步骤

# 重新构建模型（避免之前训练的权重状态干扰演示）
model = keras.Sequential([
    layers.Dense(512, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax")
])
optimizer = keras.optimizers.RMSprop(learning_rate=1e-3)
loss_fn = keras.losses.SparseCategoricalCrossentropy()
train_acc_metric = keras.metrics.SparseCategoricalAccuracy()
val_acc_metric = keras.metrics.SparseCategoricalAccuracy()


@tf.function
def train_step(x, y):
    """
    编译后的单步训练函数。
    
    这个函数会被 TensorFlow 编译成计算图：
    - 第一次调用时：追踪操作，构建图，执行
    - 后续调用时：直接使用编译好的图（速度快）
    """
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss_value = loss_fn(y, logits)
    grads = tape.gradient(loss_value, model.trainable_weights)
    optimizer.apply_gradients(zip(grads, model.trainable_weights))
    train_acc_metric.update_state(y, logits)
    return loss_value


@tf.function
def val_step(x, y):
    """编译后的单步验证函数"""
    val_logits = model(x, training=False)
    val_acc_metric.update_state(y, val_logits)


# 训练循环（与 §9 相同结构，但使用编译后的函数）
epochs = 3
for epoch in range(epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{epochs} (使用 tf.function 加速)")
    print(f"{'='*50}")
    
    for step, (batch_x, batch_y) in enumerate(train_ds):
        loss_value = train_step(batch_x, batch_y)
        if step % 100 == 0:
            print(f"  Step {step:4d}: loss = {loss_value:.4f}")
    
    train_acc = train_acc_metric.result()
    print(f"\n  训练精度: {train_acc:.4f}")
    train_acc_metric.reset_state()
    
    for val_x, val_y in val_ds:
        val_step(val_x, val_y)
    val_acc = val_acc_metric.result()
    print(f"  验证精度: {val_acc:.4f}")
    val_acc_metric.reset_state()

### 第一阶段小结：完全自定义训练

| 组件 | 作用 |
|------|------|
| `tf.GradientTape` | 自动记录前向传播，之后可以反向传播求梯度 |
| `tape.gradient()` | 计算损失对权重的梯度 |
| `optimizer.apply_gradients()` | 根据梯度更新模型权重 |
| `@tf.function` | 将 Python 函数编译成计算图，加速执行 |
| `training=True/False` | 控制 Dropout/BN 等层的行为 |

**缺点**：完全手写循环会丢失 `fit()` 的其他功能（callbacks、分布式训练、进度条等）。

---
# 第二阶段：轻量自定义 —— 覆盖 train_step()

如果你只需要修改训练逻辑的**核心部分**（梯度计算、损失组合），
但想保留 `fit()` 的 callbacks、进度条、分布式支持，
那么覆盖 `train_step()` 是最佳选择。

## §11 在 fit() 中使用自定义训练逻辑（覆盖 train_step）

原理：
- `fit()` 内部每个 batch 都会调用 `train_step(data)`
- 覆盖这个方法，就可以完全控制每个 batch 的训练行为
- 其他一切（callbacks、进度条、分布式）仍然正常工作

In [ ]:
# 11.1 覆盖 train_step() 的完整示例

class CustomTrainStepModel(keras.Model):
    """
    自定义训练步骤的模型。
    
    优势：
    - 保留 fit() 的 callbacks、分布式训练、进度条等功能
    - 可以完全控制每个 batch 的训练逻辑
    - 比完全手写训练循环更安全、更简洁
    """
    
    def train_step(self, data):
        """
        自定义单步训练逻辑。
        这个方法会在 fit() 的每个 batch 中被自动调用。
        
        参数
        ----
        data : tuple
            从 fit() 接收的一个 batch 数据 (x, y)
        
        返回
        ----
        dict : 键是指标名称，值是指标值（会显示在进度条中）
        """
        x, y = data
        
        # 1. 前向传播
        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            # compiled_loss 会自动处理 loss + regularization
            loss = self.compiled_loss(
                y, y_pred, regularization_losses=self.losses)
        
        # 2. 计算梯度
        gradients = tape.gradient(loss, self.trainable_variables)
        
        # 3. 可选：梯度裁剪（防止梯度爆炸）
        #    这在处理 RNN/LSTM 或深层网络时特别有用
        gradients = [tf.clip_by_norm(g, 1.0) for g in gradients]
        
        # 4. 更新权重
        self.optimizer.apply_gradients(
            zip(gradients, self.trainable_variables))
        
        # 5. 更新指标（使用 compile() 时配置的 metrics）
        self.compiled_metrics.update_state(y, y_pred)
        
        # 6. 返回指标字典
        #    这些值会显示在 fit() 的进度条中
        return {m.name: m.result() for m in self.metrics}

In [ ]:
# 构建自定义模型
# 注意：用 keras.Model 的函数式 API 构建基础结构
inputs = keras.Input(shape=(28 * 28,))
x = layers.Dense(512, activation="relu")(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation="softmax")(x)

# 用自定义类包装
model = CustomTrainStepModel(inputs=inputs, outputs=outputs)

# 编译：仍然使用 compile()，和以前一模一样
model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 训练：仍然使用 fit()，callbacks 仍然可用！
# 唯一的区别是：每个 batch 内部使用了我们自定义的 train_step()
model.fit(
    x_train_small, y_train_small,
    epochs=3,
    batch_size=64,
    validation_data=(x_val, y_val)
)

### 11.2 自定义评估循环：覆盖 test_step()

与 `train_step()` 类似，`test_step()` 控制验证阶段的行为。
在 `evaluate()` 和 `validation_data` 时被调用。

In [ ]:
# 覆盖 test_step() 的示例

class CustomEvalModel(keras.Model):
    """
    自定义评估步骤的模型。
    
    在 evaluate() 和 fit(validation_data=...) 时调用。
    """
    
    def test_step(self, data):
        """
        自定义单步评估逻辑。
        """
        x, y = data
        
        # 前向传播（不训练，training=False 关闭 Dropout）
        y_pred = self(x, training=False)
        
        # 更新损失和指标
        self.compiled_loss(y, y_pred, regularization_losses=self.losses)
        self.compiled_metrics.update_state(y, y_pred)
        
        return {m.name: m.result() for m in self.metrics}


# 构建模型
inputs = keras.Input(shape=(28 * 28,))
x = layers.Dense(512, activation="relu")(inputs)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation="softmax")(x)
model = CustomEvalModel(inputs=inputs, outputs=outputs)

model.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 训练：验证阶段会调用自定义的 test_step()
model.fit(
    x_train_small, y_train_small,
    epochs=2,
    validation_data=(x_val, y_val)
)

---
# 总结

### 三种训练方式的对比与选择

| 方式 | 代码量 | 灵活性 | 保留 fit() 功能？ | 适用场景 |
|------|--------|--------|-------------------|----------|
| **内置 fit()** | 最少 | 低 | ✅ 全部 | 标准分类/回归 |
| **覆盖 train_step()** | 中等 | 中 | ✅ 全部 | 梯度裁剪、多任务加权、GAN |
| **完全自定义循环** | 最多 | 最高 | ❌ 无 | 元学习、复杂研究代码 |

### 选择建议

```text
你的需求是什么？
│
├─ 标准训练 → 内置 fit()，compile + fit 一条龙
│
├─ 需要改梯度/损失组合/梯度裁剪？
│   └─→ 覆盖 train_step()，保留 fit() 的其他功能
│
└─ 需要完全控制（如 GAN 的交替训练）？
    └─→ GradientTape + tf.function 手写循环
```

### 第7章 总回顾

**模型构建（7.2 节）**
```
简单层堆叠     → Sequential
一般项目       → 函数式 API（推荐首选）
特殊研究需求   → 模型子类化
```

**训练流程（7.3 节）**
```
compile → fit → evaluate → predict  是标准流程
自定义指标：继承 keras.metrics.Metric
自定义回调：继承 keras.callbacks.Callback
监控可视化：TensorBoard
```

**训练扩展（7.4 节）**
```
fit() 不够用时：
  轻量自定义 → 覆盖 train_step()，保留 fit() 功能
  完全自定义 → GradientTape + tf.function 手写循环
```

---
**恭喜完成第7章！接下来可以继续学习更高级的深度学习主题。**